In [1]:
from unittest import skip

import pandas as pd
import numpy as np

# Daten einlesen

In [6]:
df = pd.read_csv("../data/processed/swissvotes_processed.csv")

## Kongruenz ohne Mobilisierung (Framing nicht mehr Stimmberechtigte sondern in Bezug auf

In [12]:
df_with_positions = df.copy()
neue_spalten = {}

for col in df:
    scores = []
    for i, row in df_with_positions.iterrows():
        position = row[col]
        ja_proz = row['volkja-proz']
        if col.startswith('p-') or col.endswith('-pos'):

            # Ja-Parole oder Volksinitiative bevorzugt
            if position in [1.0, 9.0]:
                scores.append((ja_proz - 50) / 100)

            # Nein-Parole oder Gegenentwurf bevorzugt
            elif position in [2.0, 8.0]:
                scores.append((50 - ja_proz) / 100)

            # Neutral
            elif position in [3.0, 4.0, 5.0, 66.0]:
                scores.append(0.0)

            else:
                scores.append(np.nan)

    # Partei- und Positions-Spalten speichern
    if col.startswith('p-') or col.endswith('-pos'):
        neue_spalten[f'zustimmung_{col}'] = scores

df_with_positions = pd.concat(
    [df_with_positions, pd.DataFrame(neue_spalten, index=df_with_positions.index)],
    axis=1
)

In [13]:
df_with_positions.to_csv('../data/processed/df_with_positions.csv', index=False)

### Datenset für Heatmap

In [ ]:
df_with_positions = df.copy()
neue_spalten = {}

for col in df:
    scores = []
    for i, row in df_with_positions.iterrows():
        position = row[col]
        ja_proz = row['volkja-proz']
        if col.startswith('p-') or col.endswith('-pos'):

            # Ja-Parole oder Volksinitiative bevorzugt
            if position in [1.0, 9.0]:
                scores.append((ja_proz - 50) / 100)

            # Nein-Parole oder Gegenentwurf bevorzugt
            elif position in [2.0, 8.0]:
                scores.append((50 - ja_proz) / 100)

            # Neutral
            elif position in [3.0, 4.0, 5.0, 66.0]:
                scores.append(0.0)

            else:
                scores.append(np.nan)

    # Partei- und Positions-Spalten speichern
    if col.startswith('p-') or col.endswith('-pos'):
        neue_spalten[f'zustimmung_{col}'] = scores

df_with_positions = pd.concat(
    [df_with_positions, pd.DataFrame(neue_spalten, index=df_with_positions.index)],
    axis=1
)

In [15]:
kanton_cols = sorted(
    c for c in df.columns if str(c).endswith("-japroz")
)
akteure = ["br-pos"] + sorted(
    c for c in df.columns if str(c).startswith("p-")
)
kanton_labels = [c.replace("-japroz", "").upper() for c in kanton_cols]


def alignment_score(position, ja_proz):
    pos = pd.to_numeric(position, errors="coerce")
    ja = pd.to_numeric(ja_proz, errors="coerce")
    if pos not in (1.0, 2.0) or pd.isna(ja):
        return np.nan
    if pos == 1.0:
        return (ja - 50) / 100
    return (50 - ja) / 100


# --- eine Abstimmung wählen (Beispiele) ---
# zeile = df.iloc[0]
# zeile = df.loc[df["anr"] == 642.0].iloc[0]

zeile = df.loc[df["anr"] == 642.0].iloc[0]  # anpassen

rows = {}
for actor in akteure:
    pos = zeile[actor]
    rows[actor] = {
        lab: alignment_score(pos, zeile[kcol])
        for kcol, lab in zip(kanton_cols, kanton_labels)
    }

heatmap_df = pd.DataFrame(rows).T
heatmap_df.index.name = "akteur"

# optional: nur Akteure mit mindestens einem gültigen Wert
heatmap_df = heatmap_df.dropna(how="all")

heatmap_df

,AG,AI,AR,BE,BL,BS,FR,GE,GL,GR,...,SH,SO,SZ,TG,TI,UR,VD,VS,ZG,ZH
akteur,,,,,,,,,,,,,,,,,,,,,
br-pos,0.1277,0.2397,0.1078,0.0964,0.095,-0.0716,0.1706,0.0067,0.099,0.112,...,0.0722,0.1031,0.2147,0.164,0.0983,0.2233,0.1275,0.2697,0.1323,0.0206
p-eco,0.1277,0.2397,0.1078,0.0964,0.095,-0.0716,0.1706,0.0067,0.099,0.112,...,0.0722,0.1031,0.2147,0.164,0.0983,0.2233,0.1275,0.2697,0.1323,0.0206
p-edu,0.1277,0.2397,0.1078,0.0964,0.095,-0.0716,0.1706,0.0067,0.099,0.112,...,0.0722,0.1031,0.2147,0.164,0.0983,0.2233,0.1275,0.2697,0.1323,0.0206
p-evp,-0.1277,-0.2397,-0.1078,-0.0964,-0.095,0.0716,-0.1706,-0.0067,-0.099,-0.112,...,-0.0722,-0.1031,-0.2147,-0.164,-0.0983,-0.2233,-0.1275,-0.2697,-0.1323,-0.0206
p-fdp,0.1277,0.2397,0.1078,0.0964,0.095,-0.0716,0.1706,0.0067,0.099,0.112,...,0.0722,0.1031,0.2147,0.164,0.0983,0.2233,0.1275,0.2697,0.1323,0.0206
p-gem,0.1277,0.2397,0.1078,0.0964,0.095,-0.0716,0.1706,0.0067,0.099,0.112,...,0.0722,0.1031,0.2147,0.164,0.0983,0.2233,0.1275,0.2697,0.1323,0.0206
p-gps,-0.1277,-0.2397,-0.1078,-0.0964,-0.095,0.0716,-0.1706,-0.0067,-0.099,-0.112,...,-0.0722,-0.1031,-0.2147,-0.164,-0.0983,-0.2233,-0.1275,-0.2697,-0.1323,-0.0206
p-kvp,-0.1277,-0.2397,-0.1078,-0.0964,-0.095,0.0716,-0.1706,-0.0067,-0.099,-0.112,...,-0.0722,-0.1031,-0.2147,-0.164,-0.0983,-0.2233,-0.1275,-0.2697,-0.1323,-0.0206
p-lega,0.1277,0.2397,0.1078,0.0964,0.095,-0.0716,0.1706,0.0067,0.099,0.112,...,0.0722,0.1031,0.2147,0.164,0.0983,0.2233,0.1275,0.2697,0.1323,0.0206
